<a href="https://colab.research.google.com/github/sneha-siju/breast-cancer-patient-survival-classification-algorithm-comparison/blob/main/Comparison_algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("reihanenamdari/breast-cancer")

print("Path to dataset files:", path)

100%|██████████| 42.8k/42.8k [00:00<00:00, 37.8MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/reihanenamdari/breast-cancer/versions/1


In [2]:
import numpy as np
import pandas as pd
import sklearn as sklearn
import os

In [3]:
print(os.listdir(path))

['Breast_Cancer.csv']


In [4]:
file=os.path.join(path, "Breast_Cancer.csv")

Breast Cancer dataset

In [5]:
df=pd.read_csv(file)
print(df.head)

<bound method NDFrame.head of       Age   Race Marital Status T Stage  N Stage 6th Stage  \
0      68  White        Married       T1      N1       IIA   
1      50  White        Married       T2      N2      IIIA   
2      58  White       Divorced       T3      N3      IIIC   
3      58  White        Married       T1      N1       IIA   
4      47  White        Married       T2      N1       IIB   
...   ...    ...            ...      ...     ...       ...   
4019   62  Other        Married       T1      N1       IIA   
4020   56  White       Divorced       T2      N2      IIIA   
4021   68  White        Married       T2      N1       IIB   
4022   58  Black       Divorced       T2      N1       IIB   
4023   46  White        Married       T2      N1       IIB   

                  differentiate Grade   A Stage  Tumor Size Estrogen Status  \
0         Poorly differentiated     3  Regional           4        Positive   
1     Moderately differentiated     2  Regional          35        

In [6]:
df.columns

Index(['Age', 'Race', 'Marital Status', 'T Stage ', 'N Stage', '6th Stage',
       'differentiate', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status',
       'Progesterone Status', 'Regional Node Examined',
       'Reginol Node Positive', 'Survival Months', 'Status'],
      dtype='object')

Using both k-fold cross validation and hyperparameter tuning using GridSearch

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

Logistic Regression, Decision Tree, Random Forest and KNN

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

In [11]:
X=df.drop(columns=['Race','Status','Survival Months'])
y=df['Status']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

Since some of the categorical data are string and Standard Scaler do not accept string values, I am converting them to numerical values using one-hot encoding

In [12]:
from sklearn.preprocessing import OneHotEncoder

In [13]:

categorical_columns = ['Marital Status','T Stage ','N Stage','6th Stage',
       'differentiate','A Stage','Estrogen Status','Grade',
       'Progesterone Status']

encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(df[categorical_columns])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=df.index
)

final_df = pd.concat(
    [df.drop(columns=categorical_columns), encoded_df],
    axis=1
)

In [14]:
print(final_df)

      Age   Race  Tumor Size  Regional Node Examined  Reginol Node Positive  \
0      68  White           4                      24                      1   
1      50  White          35                      14                      5   
2      58  White          63                      14                      7   
3      58  White          18                       2                      1   
4      47  White          41                       3                      1   
...   ...    ...         ...                     ...                    ...   
4019   62  Other           9                       1                      1   
4020   56  White          46                      14                      8   
4021   68  White          22                      11                      3   
4022   58  Black          44                      11                      1   
4023   46  White          30                       7                      2   

      Survival Months Status  Marital Status_Divorc

In [15]:

X_encoded = final_df.drop(columns=['Race', 'Status', 'Survival Months'])
y_encoded = final_df[[ 'Status']]


X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=1)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

K-Fold Cross Validation

In [16]:
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

In [17]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
scores = cross_val_score(lr, X_encoded, y_encoded, cv=kf, scoring='accuracy')
print(f"Accuracy for each fold: {scores}")

average_accuracy = np.mean(scores)
print(f"Average Accuracy: {average_accuracy:.2f}")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exam

Accuracy for each fold: [0.86335404 0.85217391 0.8621118  0.8447205  0.84701493]
Average Accuracy: 0.85


In [18]:
rf= RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
scores = cross_val_score(rf, X_encoded, y_encoded, cv=kf, scoring='accuracy')

print(f"Accuracy for each fold: {scores}")

average_accuracy = np.mean(scores)
print(f"Average Accuracy: {average_accuracy:.2f}")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example usi

Accuracy for each fold: [0.85714286 0.84596273 0.84968944 0.8310559  0.83457711]
Average Accuracy: 0.84


In [19]:
knn= KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
scores = cross_val_score(knn, X_encoded, y_encoded, cv=kf, scoring='accuracy')
print(f"Accuracy for each fold: {scores}")

average_accuracy = np.mean(scores)
print(f"Average Accuracy: {average_accuracy:.2f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for 

Accuracy for each fold: [0.83478261 0.82360248 0.83726708 0.81614907 0.81218905]
Average Accuracy: 0.82


/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [20]:
clf = DecisionTreeClassifier(random_state=1)
clf.fit(X_train, y_train)
scores = cross_val_score(clf, X_encoded, y_encoded, cv=kf, scoring='accuracy')
print(f"Accuracy for each fold: {scores}")

average_accuracy = np.mean(scores)
print(f"Average Accuracy: {average_accuracy:.2f}")


Accuracy for each fold: [0.79254658 0.74658385 0.7826087  0.7515528  0.75870647]
Average Accuracy: 0.77


After cross-validation , it is observed that Logistic Regression and Random Forest have the best Average Accuracy among the four algorithms with 0.85 and 0.84 respectively.

Next we are going to tune the hyperparameters of these two models using GridSeachCV

In [21]:

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import make_scorer, f1_score
dead_f1 = make_scorer(f1_score, pos_label='Dead')
scoring=dead_f1

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lr)
])

lr_params = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs']
}

lr_grid = GridSearchCV(lr_pipe, lr_params, cv=5, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train, y_train)

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rf_grid = GridSearchCV(
    rf,
    rf_params, cv=5, scoring='accuracy', n_jobs=-1
)
rf_grid.fit(X_train, y_train)


for name, grid in [('Logistic Regression', lr_grid), ('Random Forest', rf_grid)]:
    print(f"\n{name}")
    print("Best parameters:", grid.best_params_)
    print("Best CV accuracy:", round(grid.best_score_, 4))

    y_pred = grid.best_estimator_.predict(X_test)
    print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Logistic Regression
Best parameters: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV accuracy: 0.8509
Test accuracy: 0.8745
              precision    recall  f1-score   support

       Alive       0.88      0.98      0.93       699
        Dead       0.59      0.15      0.24       106

    accuracy                           0.87       805
   macro avg       0.74      0.57      0.59       805
weighted avg       0.85      0.87      0.84       805

Confusion matrix:
 [[688  11]
 [ 90  16]]

Random Forest
Best parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 300}
Best CV accuracy: 0.8475
Test accuracy: 0.8758
              precision    recall  f1-score   support

       Alive       0.88      0.99      0.93       699
        Dead       0.67      0.11      0.19       106

    accuracy                           0.88       805
   macro avg       0.77      0.55      0.56       805
weighted avg       0.85      0.88      0.84       805

Confusion

Even though we could increase the accuracies , looking at the classification report , we can see that recall is very low for classifying dead people.